# Juntar los archivos en una misma base de datos

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
dfs = [
    pd.read_csv(os.path.join(r, f)).assign(activo=f.replace('_marketstack.csv', ''))
    for r, _, files in os.walk('.')
    for f in files if f.lower().endswith('_marketstack.csv')
]

df = pd.concat(dfs, ignore_index=True)

# Limpieza de datos

In [ ]:
df.dtypes

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

In [ ]:
# Asegurar localización UTC antes de convertir a Nueva York
df["date"] = pd.to_datetime(df["date"])
if df["date"].dt.tz is None:
    df["date"] = df["date"].dt.tz_localize("UTC")

# Convertir la columna de fecha a la hora local real de Nueva York
df["date"] = df["date"].dt.tz_convert("America/New_York")

# Filtrar días hábiles oficiales del mercado (NYSE)
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(
    start_date=df["date"].min().date(), end_date=df["date"].max().date()
)
dias_habiles_nyse = schedule.index.date

es_dia_laborable = df["date"].dt.date.isin(dias_habiles_nyse)

# Horario bolsa de 09:30 a 16:00
hora_apertura = pd.to_datetime("09:30").time()
hora_cierre = pd.to_datetime("16:00").time()

es_horario_mercado = (df["date"].dt.time >= hora_apertura) & (
    df["date"].dt.time <= hora_cierre
)

# Filtrar DataFrame manteniendo exclusivamente registros en sesión abierta
df_horario_mercado = df[es_dia_laborable & es_horario_mercado].copy()

# Remover zona horaria manteniendo la hora local de NY
df_horario_mercado["date"] = df_horario_mercado["date"].dt.tz_localize(None)
df_horario_mercado

In [ ]:
missings = df_horario_mercado.isna().sum()
print("Missings por columna:", missings)
sns.heatmap(df_horario_mercado.isna(), cbar=False)
plt.show()

In [ ]:
# Agrupar días reales únicos por activo
df_horario_mercado["fecha_dia"] = df_horario_mercado["date"].dt.date
dias_reales = df_horario_mercado.groupby("activo")["fecha_dia"].unique()

# Identificar días laborables faltantes entre min y max de cada activo
dias_faltantes = [
    {"activo": activo, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for activo, fechas in dias_reales.items()
    for dia in set(pd.bdate_range(min(fechas), max(fechas)).date) - set(fechas)
]

df_dias_faltantes = pd.DataFrame(dias_faltantes).sort_values(
    ["activo", "fecha_faltante"]
)
df_dias_faltantes

In [ ]:
print(df_horario_mercado.groupby("activo")["date"].agg(["min", "max"]))

In [ ]:
# Agrupar por activo y día
jornadas = (
    df_horario_mercado.groupby(["activo", df_horario_mercado["date"].dt.date])
    .agg(Apertura=("date", "min"), Cierre=("date", "max"))
    .reset_index()
)

# Filtrar aperturas distintos a 09:30 o cierres distintos a 16:00 y guardar en DataFrame
df_jornadas_irregulares = jornadas[
    (jornadas["Apertura"].dt.strftime("%H:%M") != "09:30")
    | (jornadas["Cierre"].dt.strftime("%H:%M") != "16:00")
].copy()

df_jornadas_irregulares["Fecha"] = df_jornadas_irregulares["Apertura"].dt.strftime("%Y-%m-%d")
df_jornadas_irregulares["Duracion_Horas"] = (
    (df_jornadas_irregulares["Cierre"] - df_jornadas_irregulares["Apertura"]).dt.total_seconds() / 3600
).round(2)
df_jornadas_irregulares["Apertura"] = df_jornadas_irregulares["Apertura"].dt.strftime("%H:%M")
df_jornadas_irregulares["Cierre"] = df_jornadas_irregulares["Cierre"].dt.strftime("%H:%M")

# Seleccionar y reordenar columnas finales
df_jornadas_irregulares = df_jornadas_irregulares[
    ["activo", "Fecha", "Apertura", "Cierre", "Duracion_Horas"]
]
df_jornadas_irregulares

In [ ]:
# Calcular saltos dentro de la misma jornada
df_horario_mercado["gap"] = df_horario_mercado.groupby(
    ["activo", df_horario_mercado["date"].dt.date]
)["date"].diff()

# Obtener el registro con el máximo salto por activo
res = df_horario_mercado.loc[
    df_horario_mercado.groupby("activo")["gap"].idxmax()
].copy()

res["Desde_Hora"] = (res["date"] - res["gap"]).dt.strftime("%H:%M:%S")
res["Hasta_Hora"] = res["date"].dt.strftime("%H:%M:%S")
res["Fecha"] = res["date"].dt.strftime("%Y-%m-%d")
res["Minutos"] = (res["gap"].dt.total_seconds() / 60).round(1)
print("\nMAYOR SALTO INTRADÍA POR ACTIVO (DESDE / HASTA)")
print(
    res[["activo", "Fecha", "Desde_Hora", "Hasta_Hora", "Minutos"]]
    .sort_values("Minutos", ascending=False)
    .to_string(index=False)
)

In [ ]:
# Porcentaje de nulos en close por activo
df_nulos_close = (
    df_horario_mercado.groupby("activo")["close"]
    .apply(lambda x: (x.isnull().sum() / len(x)) * 100)
    .round(2)
    .reset_index(name="pct_nulos_close")
    .sort_values("pct_nulos_close", ascending=False)
)

print(df_nulos_close.to_string(index=False))

In [ ]:
fig = px.line(
    df_horario_mercado,
    x="date",
    y="close",
    color="activo",
    title="Evolución del Precio de Cierre (Close) por Activo - Detección de Splits",
    labels={"date": "Fecha", "close": "Precio de Cierre (USD)", "activo": "Ticker"},
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Precio de Cierre ($)",
    hovermode="x unified",
    legend_title_text="Activo",
    height=650,
    template="plotly_white"
)

fig.show()

In [ ]:
# Recuperar Días Completos
if not df_dias_faltantes.empty:
    tickers = df_dias_faltantes["activo"].unique().tolist()
    f_min = df_dias_faltantes["fecha_faltante"].min()
    f_max = (
        pd.to_datetime(df_dias_faltantes["fecha_faltante"].max())
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    df_yf_dias = (
        yf.download(
            tickers, start=f_min, end=f_max, interval="1d", group_by="ticker"
        )
        .stack(level=0, future_stack=True)
        .reset_index()
    )
    df_yf_dias = df_yf_dias.rename(
        columns={
            "Date": "date",
            "Ticker": "activo",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    df_yf_dias["date"] = pd.to_datetime(df_yf_dias["date"])

    # Filtrar solo las fechas que faltaban
    df_dias_rec = pd.merge(
        df_dias_faltantes,
        df_yf_dias,
        left_on=["activo", "fecha_faltante"],
        right_on=["activo", df_yf_dias["date"].dt.strftime("%Y-%m-%d")],
    ).drop(columns=["fecha_faltante"])
else:
    df_dias_rec = pd.DataFrame()

# Unificar y limpiar en el DataFrame final
cols = ["activo", "date", "open", "high", "low", "close", "volume"]
df_consolidado_completo = (
    pd.concat([df_horario_mercado[cols], df_dias_rec[cols]], ignore_index=True)
    .drop_duplicates(subset=["activo", "date"])
    .sort_values(["activo", "date"])
    .reset_index(drop=True)
)
df_consolidado_completo

In [ ]:
# Unificar formato de fecha
df_consolidado_completo["date"] = pd.to_datetime(
    df_consolidado_completo["date"]
).dt.tz_localize(None)
df_consolidado_completo["fecha_dia"] = df_consolidado_completo["date"].dt.date

# Días laborables faltantes
nyse = mcal.get_calendar("NYSE")
faltantes = [
    {"activo": activo, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for activo, g in df_consolidado_completo.groupby("activo")
    for dia in set(
        nyse.schedule(
            start_date=g["fecha_dia"].min(), end_date=g["fecha_dia"].max()
        ).index.date
    )
    - set(g["fecha_dia"].unique())
]
df_dias_faltantes_post = pd.DataFrame(faltantes)

# Jornadas irregulares
jornadas = (
    df_consolidado_completo.groupby(["activo", "fecha_dia"])
    .agg(
        apertura=("date", lambda x: x.min().strftime("%H:%M")),
        cierre=("date", lambda x: x.max().strftime("%H:%M")),
    )
    .reset_index()
)

df_jornadas_irregulares_post = jornadas[
    (jornadas["apertura"] != "09:30") | (jornadas["cierre"] != "16:00")
].reset_index(drop=True)

# Resumen
print(f"Días faltantes restantes: {len(df_dias_faltantes_post)}")
print(f"Jornadas irregulares restantes: {len(df_jornadas_irregulares_post)}")

In [ ]:
# Gráfico de líneas de close por activo
fig = px.line(
    df_consolidado_completo.sort_values(["activo", "date"]),
    x="date",
    y="close",
    color="activo",
    title="Evolución Histórica de Cierre (Close) por Activo",
    labels={"date": "Fecha", "close": "Precio Cierre (USD)", "activo": "Ticker"},
    template="plotly_white",
)

fig.update_layout(hovermode="x unified", height=500)
fig.show()

In [ ]:
# Calendario y rangos
f_ini, f_fin = "2018-06-06", "2025-06-06"
dias_nyse = set(mcal.get_calendar("NYSE").schedule(f_ini, f_fin).index.date)

df_consolidado_completo["date"] = pd.to_datetime(
    df_consolidado_completo["date"]
).dt.tz_localize(None)
r = df_consolidado_completo.groupby("activo")["date"].agg(["min", "max"])

# Descargar tramos faltantes
cols = ["activo", "date", "open", "high", "low", "close", "volume"]
dfs = [df_consolidado_completo[cols]]


def get_yf(tk, s, e):
    try:
        d = yf.download(
            tk, start=s, end=e, interval="1d", progress=False
        ).reset_index()
        if d.empty or "Date" not in d.columns:
            return
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)

        # Normalizar fecha
        d["date"] = pd.to_datetime(d["Date"])
        d["date"] = (
            d["date"].dt.tz_localize(None)
            if d["date"].dt.tz is None
            else d["date"]
            .dt.tz_convert("UTC")
            .dt.tz_convert("America/New_York")
            .dt.tz_localize(None)
        )

        d["activo"] = tk
        dfs.append(
            d.rename(
                columns={
                    "Open": "open",
                    "High": "high",
                    "Low": "low",
                    "Close": "close",
                    "Volume": "volume",
                }
            )[cols]
        )
    except Exception:
        pass


for tk, row in r.iterrows():
    if row["min"].strftime("%Y-%m-%d") > f_ini:
        get_yf(tk, f_ini, row["min"].strftime("%Y-%m-%d"))
    if row["max"].strftime("%Y-%m-%d") < f_fin:
        get_yf(tk, row["max"].strftime("%Y-%m-%d"), f_fin)

# Limpiar
df_consolidado_completo = (
    pd.concat(dfs, ignore_index=True)
    .query("date.dt.date in @dias_nyse")
    .drop_duplicates(["activo", "date"])
    .sort_values(["activo", "date"])
    .reset_index(drop=True)
)

print(df_consolidado_completo.groupby("activo")["date"].agg(["min", "max"]))
df_consolidado_completo.to_csv("dow_jones_base_completa.csv", index=False)

In [ ]:
# Asegurar orden cronológico y formato correcto
df_consolidado_completo = df_consolidado_completo.sort_values(
    ["activo", "date"]
)

# Gráfico de líneas de close por activo
fig = px.line(
    df_consolidado_completo,
    x="date",
    y="close",
    color="activo",
    title="Evolución Histórica de Cierre (Close) por Activo (Desde 2018-06-06)",
    labels={
        "date": "Fecha",
        "close": "Precio Cierre (USD)",
        "activo": "Ticker",
    },
    template="plotly_white",
)

fig.update_layout(hovermode="x unified", height=550)
fig.show()